## Import Liabrary

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook_connected"

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import load_time_indexed_csv, run_cleaning_pipeline, save_cleaned_csv

In [3]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_Trimed.csv"
OUTPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped.csv"

In [4]:
df = load_time_indexed_csv(INPUT_PATH)
df.shape

(33696, 28)

In [5]:
results = run_cleaning_pipeline(df)

df_final = results["df_final"]
events_all = results["events_all"]
reports = results["reports"]
debug = results["debug"]
spike_table = results["spike_table"]
summary_df = results["summary_removed"]

In [6]:
print("Final shape:", df_final.shape)
print("Total NaNs:", int(df_final.isna().sum().sum()))
print(summary_df.to_string(index=False))
print(spike_table.head(20).to_string(index=False))

Final shape: (33696, 9)
Total NaNs: 15232
     group  cols  raw_non_nan_points  new_nan_points_added  removed_%
      BESS     6              200196                  8865   4.428160
       ELX     1               33366                  1147   3.437631
PV_WEATHER     4              133464                  4224   3.164898
  BUILDING     4              126693                    94   0.074195
                        column  rows  non_nan_before  spikes_detected  new_nans_introduced  spikes_%_of_rows  spikes_%_of_non_nan   jump_thr  neighbor_close_thr  outer_close_thr
               PV_WS_Radiation 33696           32330               33                   33          0.097934             0.102072 511.343500              54.610           54.610
                 PV_WS_AirTemp 33696           32330               31                   31          0.091999             0.095886   9.000000               5.000            5.000
        BU_TotActPwr_Tech_Room 33696           33366               30     

In [7]:
save_cleaned_csv(df_final, OUTPUT_PATH)
print("Saved to:", OUTPUT_PATH)

Saved to: C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped.csv


## Read Data

In [8]:
path = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_Trimed.csv"
df = pd.read_csv(path,sep=',')
df.head(5)

,Time,ELX_AuxPowCons,BA_Unitstate,BA_PwrAt,BA_Soc,BU_PwrReq,BU_Unitstate,BA_PwrAtChrLimTot,BA_PwrAtDisLimTot,PV_Unitstate,...,EL_TotActPwr_NitrogenUnit,ELX_TotActPwr_Pump_Room,BU_TotActPwr_SDB_EL_Substation,BU_TotActPwr_Tech_Room,BU_TotActPwr_UPS1,BU_TotActPwr_UPS2,PV_WS_AirTemp,PV_WS_Radiation,PV_WS_RelAirPre,PV_WS_RelHum
0,2025-10-15 00:00:00,0.0,0.0,49.0,83.4,29.861,2.0,3192.34,4065.34,NaN,...,14.053,NaN,NaN,3.614,2.942,NaN,129.0,-4.6,1011.7,83.5
1,2025-10-15 00:05:00,0.0,0.0,48.0,83.4,28.847,2.0,3190.54,4066.04,NaN,...,13.983,NaN,NaN,3.608,2.960,NaN,129.0,-4.3,1011.7,83.8
2,2025-10-15 00:10:00,0.0,0.0,39.0,83.3,19.074,2.0,3191.94,4064.64,NaN,...,4.359,NaN,NaN,3.646,2.856,NaN,127.0,-4.7,1011.7,84.1
3,2025-10-15 00:15:00,0.0,0.0,48.0,83.2,27.255,2.0,3191.34,4065.24,NaN,...,14.063,NaN,NaN,3.601,2.949,NaN,127.0,-4.5,1011.6,84.3
4,2025-10-15 00:20:00,0.0,0.0,47.0,83.1,27.186,2.0,3191.34,4065.24,NaN,...,13.976,NaN,NaN,3.646,2.957,NaN,126.0,-4.5,1011.7,85.0


##### Set index

In [9]:
df['Time'] = pd.to_datetime(df['Time'], errors = 'coerce')
df.set_index('Time', inplace=True)

## Missing Value check

In [10]:
#### Missing Value check

# missing values summary by ech features (count and percent)
missing_count = df_final.isnull().sum()
missing_pct = (df_final.isnull().mean() * 100).round(2)
missing_summary = pd.concat([missing_count, missing_pct], axis=1)
missing_summary.columns = ['missing_count', 'missing_pct']
missing_summary = missing_summary.sort_values('missing_count', ascending=False)

if missing_summary['missing_count'].sum() == 0:
    print("No missing values found.")
else:
    print(missing_summary[missing_summary['missing_count'] > 0])

    # number of rows with any missing value and a sample of such rows
    n_rows_with_missing = df.isnull().any(axis=1).sum()
    print(f"\nRows with any missing values: {n_rows_with_missing}")


                                missing_count  missing_pct
BU_TotActPwr_SDB_EL_Substation           7121        21.13
BA_Soc                                   2551         7.57
PV_WS_Radiation                          1399         4.15
PV_WS_AirTemp                            1397         4.15
PV_WS_RelHum                             1382         4.10
BU_TotActPwr_Tech_Room                    360         1.07
BU_TotActPwr_Academy                      357         1.06
BA_TotActPwr_BESS_AC_Panel1               334         0.99
BA_TotActPwr_BESS_AC_Panel2               331         0.98

Rows with any missing values: 11125


In [15]:
df_final.shape

(33696, 9)

#### Histogram 

In [11]:
# num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# num_cols = [c for c in num_cols if not df[c].isna().all()]
# n = len(num_cols)
# if n == 0:
#     print("No numeric feature columns found to plot.")
# else:
#     ncols = 4 
#     nrows = math.ceil(n / ncols)

#     fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows))
#     axes = np.array(axes).reshape(-1)

#     for i, col in enumerate(num_cols):
#         x = df[col].dropna()
#         axes[i].hist(x, bins=50)
#         axes[i].set_title(col)
#         axes[i].set_ylabel("Count")

#     # Turn off empty subplots
#     for j in range(n, len(axes)):
#         axes[j].axis("off")

#     plt.tight_layout()
#     plt.show()

## Ploting Function for plotly

In [12]:
def plot_feature(df_final, cols=None, start=None, end=None, connectgaps=False):
    d = df_final.copy()

    # ensure datetime index
    d.index = pd.to_datetime(d.index, errors="coerce")
    d = d[~d.index.isna()].sort_index()

    # time filter
    if start is not None or end is not None:
        d = d.loc[start:end]

    # choose columns
    if cols is None:
        cols = d.columns.tolist()

    # numeric convert (important if some columns are strings)
    d[cols] = d[cols].apply(pd.to_numeric, errors="coerce")

    # plot multiple columns
    fig = px.line(d, x=d.index, y=cols,)
    fig.update_traces(connectgaps=connectgaps)
    fig.show()


In [13]:
# plot_feature(df,"BA_PwrAtChrLimTot")
# plot_feature(df,"BA_PwrAtDisLimTot")
# plot_feature(df,"BA_TotActPwr_BESS_AC_Panel2")
# plot_feature(df,"BA_TotActPwr_BESS_AC_Panel1")
# plot_feature(df,"BA_PwrAt")
# plot_feature(df,"BA_Unitstate")
plot_feature(df_final,"BA_Soc")